# test-A — C1: standard pipeline: MarkItDown + 1,000-character splitter

answering: **gpt-4o-mini**, embedding: **text-embedding-3-small**.

Means over 3 repeats of 150 in-scope and 20 out-of-scope questions. Reads `answers-r*.json` and `judge-r*.json` in this folder; nothing here calls an LLM.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "code"))  # test/code
import imka_eval as E

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RUN = "test-A"
rows = E.run_rows(RUN)
in_scope = rows[rows.group != "out_of_scope"]

## Documents

The five documents the run searched (pinned in `documents.json`).

In [2]:
docs = E._read_json(E.run_dir(RUN) / "documents.json")["documents"]
pd.DataFrame([{"document": d["document_name"][:45], "document_id": d["document_id"], "chunks": d.get("chunk_count"), "captioning": (d.get("models") or {}).get("captioning", "-"), "embedding": (d.get("embedding") or {}).get("model", E.DEFAULT_EMBEDDING)} for d in docs])

,document,document_id,chunks,captioning,embedding
0,ABB Manual for Induction Motors and Generator,713054b0-29cd-42b5-a55d-43b46ef5fba9,438,-,text-embedding-3-small
1,EMSA Diesel Generator manual-en.pdf,42b3313c-14e4-43f8-ac2e-9a3380dfaf75,135,-,text-embedding-3-small
2,Kohler Generator Manual.pdf,6988c3b4-5daf-43e2-aeec-b7f3c878cd61,428,-,text-embedding-3-small
3,Nidec Motor Manual.pdf,771c25ba-6f96-4ae7-be58-edb897062094,172,-,text-embedding-3-small
4,Siemens_Manual_Vertical_Induction_Motors_EN.p,8dbee7fc-536c-4a2b-8d87-6d531143e195,165,-,text-embedding-3-small


## Results by question group

In [3]:
E.summary(runs=(RUN,), by="group")

n  hit@1  hit@3  hit@5  reciprocal_rank  correct_pdf@5  correct_pdf@1  candidates_hit  faithfulness  answer_relevancy  context_recall  answer_accuracy  refused
run    group                                                                                                                                                                           
test-A tables          30  0.789  0.844  0.867            0.819          0.967          0.933           0.956         0.704             0.645           0.578            0.572      NaN
       cross_document  30  0.722  0.822  0.844            0.775          0.967          0.800           0.967         0.778             0.766           0.801            0.756      NaN
       topical         30  0.667  0.878  0.922            0.777          0.967          0.956           0.933         0.774             0.776           0.857            0.794      NaN
       paraphrase      35  0.619  0.752  0.790            0.690          0.952          0.857           0.933         0.736             0.687           0.686            0.574      NaN
       figures         25  0.413  0.453  0.533            0.450          0.907          0.813           0.613         0.867             0.201           0.197            0.167      NaN
       out_of_scope    20    NaN    NaN    NaN              NaN            NaN            NaN             NaN           NaN               NaN             NaN              NaN      1.0

In [4]:
E.summary(runs=(RUN,), by=None)

,n,hit@1,hit@3,hit@5,reciprocal_rank,correct_pdf@5,correct_pdf@1,candidates_hit,faithfulness,answer_relevancy,context_recall,answer_accuracy,refused
run,,,,,,,,,,,,,
test-A,170,0.649,0.76,0.8,0.71,0.953,0.873,0.891,0.767,0.631,0.64,0.586,1.0


## Stability across the three repeats

In [5]:
in_scope.groupby("repeat")[["hit@5", "context_recall", "answer_accuracy"]].mean().round(3)

,hit@5,context_recall,answer_accuracy
repeat,,,
1,0.807,0.661,0.587
2,0.787,0.631,0.577
3,0.807,0.628,0.595


## Out-of-scope questions

Share of the 20 out-of-scope questions declined, per repeat.

In [6]:
rows[rows.group == "out_of_scope"].groupby("repeat")["refused"].mean().round(3)

repeat
1    1.0
2    1.0
3    1.0
Name: refused, dtype: float64

## Lowest answer accuracy

The ten in-scope questions with the lowest accuracy, averaged over the repeats, with the first repeat's answer.

In [7]:
answers = {r["question"]: r.get("answer") or "" for r in E.load_answers(RUN, 1)["results"]}
worst = (in_scope.groupby(["question", "group", "document_name"])["answer_accuracy"].mean()
         .sort_values().head(10).reset_index())
worst["document_name"] = worst["document_name"].str[:20]
worst["answer (repeat 1)"] = worst["question"].map(lambda q: answers.get(q, "")[:160])
worst

,question,group,document_name,answer_accuracy,answer (repeat 1)
0,According to the Nidec U.S. MOTORS product ser...,topical,Nidec Motor Manual.p,0.0,The inspection of a motor after extended stora...
1,"As a thumb rule, how often does ABB say an ins...",topical,ABB Manual for Induc,0.0,The context does not specify how often an insu...
2,"After re-greasing a U.S. MOTORS brand motor, h...",cross_document,Nidec Motor Manual.p,0.0,The context does not specify how long a U.S. M...
3,According to the Siemens typical motor control...,tables,Siemens_Manual_Verti,0.0,The context does not provide specific alarm an...
4,"For ABB flood-lubricated sleeve bearings, what...",paraphrase,ABB Manual for Induc,0.0,The normal oil feed pressure for ABB flood-lub...
5,"For an ABB induction machine, what is the tigh...",tables,ABB Manual for Induc,0.0,The context does not provide the tightening to...
6,At what winding temperatures should a Siemens ...,paraphrase,Siemens_Manual_Verti,0.0,The context does not provide specific alarm an...
7,EMSA generator battery: which hydrometer speci...,paraphrase,EMSA Diesel Generato,0.0,The context does not specify which hydrometer ...
8,How many litres of lubricating oil does an oil...,paraphrase,Kohler Generator Man,0.0,The context does not specify the quantity of l...
9,How many newton-metres should an oiled M20 gra...,paraphrase,ABB Manual for Induc,0.0,An oiled M20 grade 8.8 bolt should be torqued ...


## Answering cost and time per question

OpenAI models are priced per token; Qwen per second of model working time (llama-server's prompt + generation time) at `E.GPU_PRICE_PER_HOUR`. RunPod queueing, cold starts and job overhead are shown (`runpod_exec_s`, `runpod_delay_s`) but not charged. `wall_s` is the question's end-to-end time as the evaluation saw it.

In [8]:
E.answering_cost((RUN,)).T

run,test-A
answers,510
calls/q,3.0
prompt_tokens/q,2296
completion_tokens/q,156
model_s/q,None
runpod_exec_s/q,None
runpod_delay_s/q,None
wall_s/q (median),4.2
"cost per 1,000 questions ($)",0.44


## Judge spend (Anthropic)

In [9]:
spend = E.judge_spend((RUN,))
spend.groupby("model")[["calls", "input_tokens", "output_tokens", "cost_usd"]].sum()

,calls,input_tokens,output_tokens,cost_usd
model,,,,
claude-haiku-5-5,2760,5523149,687024,0.8959
claude-sonnet-5-5,900,1137996,9000,2.3659
